<a href="https://colab.research.google.com/github/Priyaa1904/Flyrank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Priyaa1904/Flyrank-ML-Internship-Starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. Signal checks and my rule

Before defining the baseline rule, I check two signals that the rule will rely on.

### Signal 1 — Staleness

**Signal:** `days_since_last_update`

**Why:** Staleness is directly related to the Content Refresh action and is a signal behind FlyRank's refresh flags.

**Verdict: CONFIRMED**

The bucket distribution shows meaningful variation in staleness, giving the rule a useful way to distinguish recently updated pages from substantially older pages.


### Signal 2 — Search visibility

**Signal:** `impressions_90d`

**Why:** Search impressions indicate how much search visibility a page has received. A refresh is more actionable when a page has demonstrated some search visibility.

**Verdict: CONFIRMED**

The bucket distribution spans all five visibility levels, so impressions provide useful differentiation for prioritizing content-refresh candidates.
### Baseline rule

The baseline will combine staleness and search visibility into one deterministic action score. Higher scores will prioritize pages that are both older since their last update and have meaningful search visibility.

The rule will produce:
- one numeric score,
- one reason code,
- one action label.

This is a simple decision-support baseline, not a trained model.

In [15]:
import pandas as pd
import numpy as np

# Load the starter Content Refresh dataset
DATA_PATH = "/content/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Rows:", len(df))
print("Columns:", len(df.columns))

# -----------------------------
# Signal 1: Staleness
# -----------------------------

staleness_bins = [-1, 30, 60, 90, 180, 365, np.inf]
staleness_labels = [
    "0-30 days",
    "31-60 days",
    "61-90 days",
    "91-180 days",
    "181-365 days",
    "365+ days"
]

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=staleness_bins,
    labels=staleness_labels
)

staleness_table = (
    df["staleness_bucket"]
    .value_counts(sort=False, dropna=False)
    .rename_axis("staleness_bucket")
    .reset_index(name="n")
)

print("\nSignal 1 — Staleness")
print(staleness_table)

# -----------------------------
# Signal 2: Search visibility
# -----------------------------

visibility_bins = [-1, 10, 100, 1000, 10000, np.inf]
visibility_labels = [
    "0-10 impressions",
    "11-100 impressions",
    "101-1K impressions",
    "1K-10K impressions",
    "10K+ impressions"
]

df["visibility_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=visibility_bins,
    labels=visibility_labels
)

visibility_table = (
    df["visibility_bucket"]
    .value_counts(sort=False, dropna=False)
    .rename_axis("visibility_bucket")
    .reset_index(name="n")
)

print("\nSignal 2 — Search visibility")
print(visibility_table)

Rows: 30000
Columns: 44

Signal 1 — Staleness
  staleness_bucket      n
0        0-30 days  20480
1       31-60 days    128
2       61-90 days     47
3      91-180 days   9171
4     181-365 days    169
5        365+ days      5

Signal 2 — Search visibility
    visibility_bucket     n
0    0-10 impressions  3879
1  11-100 impressions  4127
2  101-1K impressions  8485
3  1K-10K impressions  9907
4    10K+ impressions  3602


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline scoring rule

I use a simple deterministic score to prioritize Content Refresh candidates.

The score combines:

1. **Staleness:** older content receives a higher score.
2. **Search visibility:** content with higher 90-day impressions receives a higher score.

Each signal contributes 0–2 points:

- Staleness:
  - 0–90 days → 0 points
  - 91–180 days → 1 point
  - 181+ days → 2 points

- Search visibility:
  - 0–100 impressions → 0 points
  - 101–10,000 impressions → 1 point
  - 10,000+ impressions → 2 points

The final baseline score is:

`staleness_score + visibility_score`

A page is assigned the action `REFRESH` when its score is at least 2.

The single reason code is `STALE_HIGH_VISIBILITY`.

This rule is intentionally simple and transparent so that a future ML model can be compared against it.

In [16]:
# -----------------------------
# Baseline Action Score
# -----------------------------

# Staleness score
df["staleness_score"] = np.select(
    [
        df["days_since_last_update"].between(0, 90),
        df["days_since_last_update"].between(91, 180),
        df["days_since_last_update"] >= 181
    ],
    [0, 1, 2],
    default=0
)

# Visibility score
df["visibility_score"] = np.select(
    [
        df["impressions_90d"].between(0, 100),
        df["impressions_90d"].between(101, 10000),
        df["impressions_90d"] >= 10001
    ],
    [0, 1, 2],
    default=0
)

# Combined score
df["action_score"] = (
    df["staleness_score"] +
    df["visibility_score"]
)

# ONE reason code
df["reason_code"] = np.where(
    df["action_score"] >= 2,
    "STALE_HIGH_VISIBILITY",
    "LOW_PRIORITY"
)

# Action label
df["action_label"] = np.where(
    df["action_score"] >= 2,
    "REFRESH",
    "MONITOR"
)

# Rank highest-priority pages first
baseline_queue = (
    df.sort_values(
        ["action_score", "impressions_90d"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

print("Baseline queue created.")
print("Rows:", len(baseline_queue))

baseline_queue[
    [
        "content_id",
        "client_id",
        "days_since_last_update",
        "impressions_90d",
        "action_score",
        "reason_code",
        "action_label"
    ]
].head(10)

Baseline queue created.
Rows: 30000


,content_id,client_id,days_since_last_update,impressions_90d,action_score,reason_code,action_label
0,content_cf56e2e2e282,client_7f2253d7e2,194,61678,4,STALE_HIGH_VISIBILITY,REFRESH
1,content_7368877ea310,client_7f2253d7e2,194,59472,4,STALE_HIGH_VISIBILITY,REFRESH
2,content_1bfaa38ff26c,client_7f2253d7e2,194,25715,4,STALE_HIGH_VISIBILITY,REFRESH
3,content_0a91db491d14,client_7f2253d7e2,193,13299,4,STALE_HIGH_VISIBILITY,REFRESH
4,content_5fe46e04994d,client_4e07408562,104,517715,3,STALE_HIGH_VISIBILITY,REFRESH
5,content_2dba2b1f9536,client_6208ef0f77,104,443434,3,STALE_HIGH_VISIBILITY,REFRESH
6,content_2c2606c5d176,client_19581e27de,104,347399,3,STALE_HIGH_VISIBILITY,REFRESH
7,content_cb112fce36be,client_19581e27de,104,309910,3,STALE_HIGH_VISIBILITY,REFRESH
8,content_9532f197bbc8,client_4e07408562,104,309192,3,STALE_HIGH_VISIBILITY,REFRESH
9,content_36ff89c8214e,client_19581e27de,104,295097,3,STALE_HIGH_VISIBILITY,REFRESH


In [17]:
import os

os.makedirs("work/outputs", exist_ok=True)

queue_output = baseline_queue[
    [
        "content_id",
        "client_id",
        "days_since_last_update",
        "impressions_90d",
        "action_score",
        "reason_code",
        "action_label"
    ]
]

queue_output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved:")
print("work/outputs/baseline_action_score.csv")
print("Rows written:", len(queue_output))

Saved:
work/outputs/baseline_action_score.csv
Rows written: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Skeptical review of the top 10

The following review checks whether each high-priority recommendation could be wrong despite receiving a high baseline score.

For every item, I record the recommended action, why the rule selected it, and what evidence could make the recommendation wrong.

In [18]:
# Build the top-10 review table

top10 = baseline_queue.head(10).copy()

top10_review = top10[
    [
        "content_id",
        "client_id",
        "days_since_last_update",
        "impressions_90d",
        "action_score",
        "reason_code",
        "action_label"
    ]
].copy()

top10_review["why_its_here"] = (
    "High baseline score from staleness and search visibility"
)

top10_review["what_would_make_it_wrong"] = (
    "The page may already be performing well, may have an update planned, "
    "or may not be appropriate for refresh despite the rule signals."
)

top10_review

,content_id,client_id,days_since_last_update,impressions_90d,action_score,reason_code,action_label,why_its_here,what_would_make_it_wrong
0,content_cf56e2e2e282,client_7f2253d7e2,194,61678,4,STALE_HIGH_VISIBILITY,REFRESH,High baseline score from staleness and search ...,"The page may already be performing well, may h..."
1,content_7368877ea310,client_7f2253d7e2,194,59472,4,STALE_HIGH_VISIBILITY,REFRESH,High baseline score from staleness and search ...,"The page may already be performing well, may h..."
2,content_1bfaa38ff26c,client_7f2253d7e2,194,25715,4,STALE_HIGH_VISIBILITY,REFRESH,High baseline score from staleness and search ...,"The page may already be performing well, may h..."
3,content_0a91db491d14,client_7f2253d7e2,193,13299,4,STALE_HIGH_VISIBILITY,REFRESH,High baseline score from staleness and search ...,"The page may already be performing well, may h..."
4,content_5fe46e04994d,client_4e07408562,104,517715,3,STALE_HIGH_VISIBILITY,REFRESH,High baseline score from staleness and search ...,"The page may already be performing well, may h..."
5,content_2dba2b1f9536,client_6208ef0f77,104,443434,3,STALE_HIGH_VISIBILITY,REFRESH,High baseline score from staleness and search ...,"The page may already be performing well, may h..."
6,content_2c2606c5d176,client_19581e27de,104,347399,3,STALE_HIGH_VISIBILITY,REFRESH,High baseline score from staleness and search ...,"The page may already be performing well, may h..."
7,content_cb112fce36be,client_19581e27de,104,309910,3,STALE_HIGH_VISIBILITY,REFRESH,High baseline score from staleness and search ...,"The page may already be performing well, may h..."
8,content_9532f197bbc8,client_4e07408562,104,309192,3,STALE_HIGH_VISIBILITY,REFRESH,High baseline score from staleness and search ...,"The page may already be performing well, may h..."
9,content_36ff89c8214e,client_19581e27de,104,295097,3,STALE_HIGH_VISIBILITY,REFRESH,High baseline score from staleness and search ...,"The page may already be performing well, may h..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks and limitations

The baseline is intentionally simple, so some recommendations can be wrong.

A weak pick is a page that receives a high score from the rule but may not actually be a good refresh candidate. For example, a page can be old and highly visible while already performing well, having a planned update, or being unsuitable for refresh.

These examples show why the baseline should be treated as a prioritization aid rather than a final decision-maker.

In [19]:
# Show high-scoring pages that may illustrate weaknesses of the rule

weak_picks = baseline_queue[
    (baseline_queue["action_score"] >= 3)
].head(5).copy()

weak_picks[
    [
        "content_id",
        "client_id",
        "days_since_last_update",
        "impressions_90d",
        "action_score",
        "reason_code",
        "action_label"
    ]
]

,content_id,client_id,days_since_last_update,impressions_90d,action_score,reason_code,action_label
0,content_cf56e2e2e282,client_7f2253d7e2,194,61678,4,STALE_HIGH_VISIBILITY,REFRESH
1,content_7368877ea310,client_7f2253d7e2,194,59472,4,STALE_HIGH_VISIBILITY,REFRESH
2,content_1bfaa38ff26c,client_7f2253d7e2,194,25715,4,STALE_HIGH_VISIBILITY,REFRESH
3,content_0a91db491d14,client_7f2253d7e2,193,13299,4,STALE_HIGH_VISIBILITY,REFRESH
4,content_5fe46e04994d,client_4e07408562,104,517715,3,STALE_HIGH_VISIBILITY,REFRESH


In [20]:
# Leakage check for the baseline rule

forbidden_columns = [
    "is_declining_label",
    "trend_direction",
    "trend_pct"
]

future_window_columns = [
    col for col in df.columns
    if "future" in col.lower()
    or "next" in col.lower()
    or "prev30" in col.lower()
]

used_columns = [
    "days_since_last_update",
    "impressions_90d"
]

forbidden_used = [
    col for col in forbidden_columns
    if col in used_columns
]

future_used = [
    col for col in future_window_columns
    if col in used_columns
]

print("Forbidden / label-derived columns used:")
print(forbidden_used)

print("\nFuture-window columns used:")
print(future_used)

assert forbidden_used == []
assert future_used == []

print("\nLeakage check PASSED.")

Forbidden / label-derived columns used:
[]

Future-window columns used:
[]

Leakage check PASSED.


In [21]:
# Confirm that no product-decision flags are used by the rule

product_flag_candidates = [
    col for col in df.columns
    if any(term in col.lower() for term in [
        "flag",
        "action",
        "recommend",
        "decision"
    ])
]

product_flags_used = [
    col for col in product_flag_candidates
    if col in used_columns
]

print("Product-decision columns used:")
print(product_flags_used)

assert product_flags_used == []

print("Product-flag leakage check PASSED.")

Product-decision columns used:
[]
Product-flag leakage check PASSED.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.